# 📋 監査ログ分析

YAML 形式の時刻別監査ログ（`audit/v18/YYYY-MM-DD/bushidan-hour-HH.yaml`）を分析します。

ログが少ない場合は `messages` / `routing_decisions` テーブルで代替表示します。

In [ ]:
import utils
from utils import qdf, scalar, ROLE_JA, PALETTE
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from datetime import datetime, timedelta
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

AUDIT_DIR = Path('/mnt/Bushidan-Multi-Agent/audit/v18')
print(f'監査ログディレクトリ: {AUDIT_DIR}')
print(f'存在: {AUDIT_DIR.exists()}')

## 1. YAML 監査ログ読み込み

In [ ]:
def load_audit_yaml(days: int = 30) -> pd.DataFrame:
    """直近 N 日分の監査 YAML を DataFrame に変換する"""
    entries: list[dict] = []
    cutoff = datetime.now() - timedelta(days=days)

    if not AUDIT_DIR.exists():
        return pd.DataFrame()

    for day_dir in sorted(AUDIT_DIR.iterdir()):
        if not day_dir.is_dir():
            continue
        try:
            day_date = datetime.strptime(day_dir.name, '%Y-%m-%d')
        except ValueError:
            continue
        if day_date < cutoff:
            continue

        for yaml_file in sorted(day_dir.glob('bushidan-hour-*.yaml')):
            try:
                with open(yaml_file, encoding='utf-8') as f:
                    data = yaml.safe_load(f)
                if isinstance(data, dict) and 'entries' in data:
                    for entry in data['entries']:
                        if isinstance(entry, dict):
                            entry['_date'] = day_dir.name
                            entries.append(entry)
            except Exception as e:
                print(f'  ⚠️  {yaml_file.name}: {e}')

    return pd.DataFrame(entries) if entries else pd.DataFrame()

audit_df = load_audit_yaml(days=30)
print(f'読み込み件数: {len(audit_df)}')
if len(audit_df) > 0:
    print(f'カラム: {audit_df.columns.tolist()}')
    display(audit_df.head())
else:
    print('ℹ️  監査ログがまだ存在しません。チャット利用後に /api/v18/evolve を実行すると記録されます。')

## 2. 監査ログ基本統計（YAML が存在する場合）

In [ ]:
if len(audit_df) > 0 and 'selected_role' in audit_df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    # ロール分布
    role_counts = audit_df['selected_role'].value_counts()
    role_counts.index = [ROLE_JA.get(r, r) for r in role_counts.index]
    role_counts.plot(kind='barh', ax=axes[0,0], color=PALETTE[0])
    axes[0,0].set_title('ロール別 実行件数')

    # Intent 型
    if 'intent_type' in audit_df.columns:
        audit_df['intent_type'].value_counts().plot(
            kind='pie', ax=axes[0,1], autopct='%1.1f%%', colors=PALETTE)
        axes[0,1].set_title('Intent 型分布')
        axes[0,1].set_ylabel('')

    # Complexity 分布
    if 'complexity' in audit_df.columns:
        audit_df['complexity'].value_counts().plot(
            kind='bar', ax=axes[1,0], color=PALETTE[2])
        axes[1,0].set_title('Complexity 分布')
        axes[1,0].tick_params(axis='x', rotation=20)

    # 実行時間分布
    if 'execution_time_ms' in audit_df.columns:
        ms = audit_df['execution_time_ms'].dropna().astype(float)
        axes[1,1].hist(ms, bins=25, color=PALETTE[3], edgecolor='white')
        axes[1,1].set_title('実行時間分布')
        axes[1,1].set_xlabel('ms')

    plt.suptitle('監査ログ分析', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('─' * 45)
    print('監査 YAML が蓄積されるまでの代替: DB からメッセージ統計を表示します')
    print('─' * 45)

## 3. DB ベース代替分析（全期間）

In [ ]:
# routing_decisions から分析（空の場合は messages で補完）
rd_count = scalar("SELECT COUNT(*) FROM routing_decisions")
print(f'routing_decisions: {rd_count} 件')

if rd_count > 0:
    # ルーティング決定の完全分析
    rd = qdf("""
        SELECT DATE(created_at) AS day,
               selected_role, intent_type, complexity,
               confidence, total_latency_ms
        FROM routing_decisions
        ORDER BY created_at DESC
    """)
    rd['day'] = pd.to_datetime(rd['day'])
    rd['role_ja'] = rd['selected_role'].map(lambda r: ROLE_JA.get(r, r))

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    rd['role_ja'].value_counts().plot(kind='barh', ax=axes[0,0], color=PALETTE[0])
    axes[0,0].set_title('ロール別選択回数')

    if 'intent_type' in rd.columns:
        rd['intent_type'].value_counts().plot(
            kind='pie', ax=axes[0,1], autopct='%1.1f%%', colors=PALETTE)
        axes[0,1].set_title('Intent 型')
        axes[0,1].set_ylabel('')

    rd['confidence'].hist(bins=20, ax=axes[1,0], color=PALETTE[2], edgecolor='white')
    axes[1,0].set_title('信頼度分布')

    rd['total_latency_ms'].dropna().hist(bins=20, ax=axes[1,1], color=PALETTE[3], edgecolor='white')
    axes[1,1].set_title('ルーティングレイテンシ分布')
    axes[1,1].set_xlabel('ms')

    plt.tight_layout()
    plt.show()
else:
    # messages テーブルで代替
    daily = qdf("""
        SELECT DATE(created_at) AS day,
               COUNT(*) AS total,
               COUNT(*) FILTER (WHERE role='user')      AS user,
               COUNT(*) FILTER (WHERE role='assistant') AS bot
        FROM messages
        GROUP BY DATE(created_at)
        ORDER BY day
    """)
    if utils.check_data(daily, 'メッセージ日別データ'):
        daily['day'] = pd.to_datetime(daily['day'])
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.fill_between(daily['day'], daily['user'], alpha=0.6, label='ユーザー', color=PALETTE[1])
        ax.fill_between(daily['day'], daily['bot'], alpha=0.6, label='エージェント', color=PALETTE[0])
        ax.set_title('日別メッセージ数推移')
        ax.set_ylabel('件数')
        ax.legend()
        plt.tight_layout()
        plt.show()

## 4. エラー傾向（実行時間が異常に長いケース）

In [ ]:
slow = qdf("""
    SELECT agent_role, model,
           ROUND(execution_time::numeric, 0) AS ms,
           LEFT(content, 80) AS content_preview,
           created_at
    FROM messages
    WHERE role='assistant'
      AND execution_time > (
          SELECT AVG(execution_time) + 2 * STDDEV(execution_time)
          FROM messages WHERE role='assistant' AND execution_time > 0
      )
      AND execution_time > 0
    ORDER BY execution_time DESC
    LIMIT 10
""")

print(f'外れ値（平均+2σ超）: {len(slow)} 件')
if len(slow) > 0:
    slow['役職'] = slow['agent_role'].map(lambda r: ROLE_JA.get(r, r))
    display(slow[['役職','model','ms','content_preview','created_at']].rename(
        columns={'ms':'実行時間(ms)','content_preview':'内容(先頭80文字)','created_at':'日時'}
    ).set_index('役職'))